# Academic PDF Extraction Pipeline
output folder , 
This notebook follows a hybrid pipeline for academic books:

In [31]:

from pathlib import Path
import json
import os
import re
import shutil

import fitz
from dotenv import load_dotenv

# --------------------------------------------------
# Locate Repository Root
# --------------------------------------------------

repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

pdf_path = os.getenv("PDF_PATH")

if not pdf_path:
    raise ValueError("PDF_PATH not found inside .env")

pdf_path = Path(pdf_path)

if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not pdf_path.exists():
    raise FileNotFoundError(pdf_path)

# --------------------------------------------------
# Open PDF
# --------------------------------------------------

doc = fitz.open(pdf_path)

print("=" * 60)
print(f"PDF : {pdf_path.name}")
print(f"Total Pages : {len(doc)}")
print("=" * 60)

# --------------------------------------------------
# Output Folder
# --------------------------------------------------

output_dir = repo_root / "Backend" / "Experiments" / "test3_output"
output_dir.mkdir(parents=True, exist_ok=True)

json_output = output_dir / "chapters_topics.json"

# --------------------------------------------------
# Read TOC
# --------------------------------------------------

toc = doc.get_toc()

if not toc:
    raise Exception(
        "This PDF does not contain bookmarks (TOC).\n"
        "Use heading/font detection instead."
    )

# --------------------------------------------------
# Extract Hierarchy
# --------------------------------------------------

chapters = []

current_chapter = None
current_topic = None

for i, item in enumerate(toc):

    level, title, start_page = item

    title = re.sub(r"\s+", " ", title).strip()

    if i < len(toc) - 1:
        end_page = toc[i + 1][2] - 1
    else:
        end_page = len(doc)

    # -------------------------------
    # Level 1 -> Chapter
    # -------------------------------

    if level == 1:

        current_chapter = {

            "chapter_number": len(chapters) + 1,
            "chapter_title": title,
            "start_page": start_page,
            "end_page": end_page,
            "topics": []

        }

        chapters.append(current_chapter)
        current_topic = None

    # -------------------------------
    # Level 2 -> Topic
    # -------------------------------

    elif level == 2 and current_chapter:

        current_topic = {

            "topic_number": len(current_chapter["topics"]) + 1,
            "topic_title": title,
            "start_page": start_page,
            "end_page": end_page,
            "subtopics": []

        }

        current_chapter["topics"].append(current_topic)

    # -------------------------------
    # Level 3 -> Subtopic
    # -------------------------------

    elif level == 3 and current_topic:

        current_topic["subtopics"].append({

            "subtopic_number": len(current_topic["subtopics"]) + 1,
            "subtopic_title": title,
            "start_page": start_page,
            "end_page": end_page

        })

# --------------------------------------------------
# Fix Chapter End Pages
# --------------------------------------------------

for i in range(len(chapters)):

    if i < len(chapters) - 1:
        chapters[i]["end_page"] = chapters[i + 1]["start_page"] - 1
    else:
        chapters[i]["end_page"] = len(doc)

# --------------------------------------------------
# Fix Topic & Subtopic End Pages
# --------------------------------------------------

for chapter in chapters:

    topics = chapter["topics"]

    for i, topic in enumerate(topics):

        if i < len(topics) - 1:
            topic["end_page"] = topics[i + 1]["start_page"] - 1
        else:
            topic["end_page"] = chapter["end_page"]

        subtopics = topic["subtopics"]

        for j, subtopic in enumerate(subtopics):

            if j < len(subtopics) - 1:
                subtopic["end_page"] = subtopics[j + 1]["start_page"] - 1
            else:
                subtopic["end_page"] = topic["end_page"]

# --------------------------------------------------
# JSON
# --------------------------------------------------

result = {

    "pdf_name": pdf_path.name,
    "total_pages": len(doc),
    "total_chapters": len(chapters),
    "chapters": chapters

}

with open(json_output, "w", encoding="utf-8") as f:

    json.dump(
        result,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON Created Successfully")

print(json_output) 
 

# --------------------------------------------------
# Create Folder Structure From JSON
# --------------------------------------------------

print("\nReading JSON...")

with open(json_output, "r", encoding="utf-8") as f:
    data = json.load(f)

root_folder = output_dir / "PDF_STRUCTURE"

# Delete existing folder structure
if root_folder.exists():
    shutil.rmtree(root_folder)

root_folder.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Clean Folder Names
# --------------------------------------------------

invalid_chars = '<>:"/\\|?*'


def clean(name: str) -> str:
    """
    Make folder names safe for Windows/Linux/Mac.
    """

    if not name:
        return "Untitled"

    for ch in invalid_chars:
        name = name.replace(ch, "")

    name = name.replace("\n", " ")
    name = name.replace("\r", " ")
    name = name.replace("\t", " ")

    name = re.sub(r"\s+", " ", name).strip()

    return name


# --------------------------------------------------
# Create Folder Structure
# --------------------------------------------------

print("\nCreating Folder Structure...\n")

chapter_count = 0
topic_count = 0
subtopic_count = 0

for chapter in data.get("chapters", []):

    chapter_name = f"C{chapter['chapter_number']:02d}_{clean(chapter['chapter_title'])[:35]}"

    chapter_folder = root_folder / chapter_name
    chapter_folder.mkdir(parents=True, exist_ok=True)

    chapter_count += 1

    print(f"📁 {chapter_name}")

    for topic in chapter.get("topics", []):

        topic_name = f"T{topic['topic_number']:02d}_{clean(topic['topic_title'])[:35]}"

        topic_folder = chapter_folder / topic_name
        topic_folder.mkdir(parents=True, exist_ok=True)

        topic_count += 1

        print(f"    📂 {topic_name}")

        for subtopic in topic.get("subtopics", []):

            subtopic_name = f"S{subtopic['subtopic_number']:02d}_{clean(subtopic['subtopic_title'])[:35]}"

            subtopic_folder = topic_folder / subtopic_name
            subtopic_folder.mkdir(parents=True, exist_ok=True)

            subtopic_count += 1

            print(f"        📄 {subtopic_name}")

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n" + "=" * 60)
print("PDF PROCESSING COMPLETED")
print("=" * 60)

print(f"PDF Name        : {data['pdf_name']}")
print(f"Total Pages     : {data['total_pages']}")
print(f"Chapters        : {chapter_count}")
print(f"Topics          : {topic_count}")
print(f"Subtopics       : {subtopic_count}")

print(f"\nJSON File")
print(json_output)

print(f"\nFolder Structure")
print(root_folder)

print("=" * 60)
print("Completed Successfully")
print("=" * 60)

PDF : sample_book.pdf
Total Pages : 1170

JSON Created Successfully
c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Experiments\test3_output\chapters_topics.json

Reading JSON...

Creating Folder Structure...

📁 C01_Cover
📁 C02_Title Page
📁 C03_Copyright
📁 C04_Dedication
📁 C05_About the Authors
📁 C06_Preface
    📂 T01_KEY FEATURES
    📂 T02_RETAINED AND IMPROVED FEATURES
    📂 T03_COMMITMENT TO EDUCATING STUDENTS
    📂 T04_ABBREVIATIONS
📁 C07_Acknowledgments
📁 C08_Contents
📁 C09_List of Clinical Blue Boxes
📁 C10_Figure Credits
📁 C11_Introduction to Clinically Oriented
    📂 T01_APPROACHES TO STUDYING ANATOMY
        📄 S01_Regional Anatomy
        📄 S02_Systemic Anatomy
        📄 S03_Clinical Anatomy
    📂 T02_ANATOMICOMEDICAL TERMINOLOGY
        📄 S01_Anatomical Position
        📄 S02_Anatomical Planes
        📄 S03_Terms of Relationship and Compariso
        📄 S04_Terms of Laterality
        📄 S05_Terms of Movement
    📂 T03_ANATOMICAL VARIATIONS
    📂 T04_I

#_______________________________________________________________________________________________

______________________________________________________________________________________